In [ ]:
import pandas as pd
fp = "../data/sba_loans_prepared/sba_loans_num_enc_train.csv"
df = pd.read_csv(fp)
test_loan_status = df.LoanStatus
chgoff_count = df.LoanStatus.value_counts()[1]
PIF_count = df.LoanStatus.value_counts()[0]

In [ ]:
pp_chgoff = chgoff_count/(chgoff_count + PIF_count)
pp_PIF = 1 - pp_chgoff
del df

In [ ]:
pp_PIF

In [ ]:

fp = "../data/sba_loans_prepared/sba_train_pca_bad_loans.csv"
df = pd.read_csv(fp)


In [ ]:
NUM_DIMS = 7
dims_reduced = ["PC-" + str(i+1) for i in range(NUM_DIMS)]
df = df[dims_reduced]

In [ ]:
from scipy import stats

In [ ]:
kchg_off = stats.gaussian_kde(df.values.T, bw_method="scott") # note scipy stats requires matrix to be num_dim x num_samples

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_pca_enc_chgoff_test.csv"
df_test_chgoff = pd.read_csv(fp)
df_test_chgoff = df_test_chgoff[dims_reduced]

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_pca_enc_chgoff_val.csv"
df_val_chgoff = pd.read_csv(fp)
df_val_chgoff = df_val_chgoff[dims_reduced]

In [ ]:
df_val_chgoff = pd.DataFrame(kchg_off.pdf(df_val_chgoff.values.T))

In [ ]:
df_val_chgoff.columns = ["chgoff_dens"]

In [ ]:
df_test_chgoff = pd.DataFrame(kchg_off.pdf(df_test_chgoff.values.T))

In [ ]:
df_test_chgoff.columns = ["chgoff_dens"]

In [ ]:
fp = "../data/sba_loans_prepared/sba_train_pca_good_loans.csv"
df = pd.read_csv(fp)
df = df[dims_reduced]

In [ ]:
kpif = stats.gaussian_kde(df.values.T, bw_method="scott")

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_pca_enc_good_val.csv"
df_val_good = pd.read_csv(fp)
df_val_good = df_val_good[dims_reduced]

In [ ]:
df_val_good = pd.DataFrame(kpif.pdf(df_val_good.values.T))
df_val_good.columns = ["PIF_dens"]

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_pca_enc_good_test.csv"
df_test_good = pd.read_csv(fp)
df_test_good = df_test_good[dims_reduced]

In [ ]:
df_test_good = pd.DataFrame(kpif.pdf(df_test_good.values.T))

In [ ]:
df_test_good.columns = ["PIF_dens"]

In [ ]:
df_test_eval = pd.concat([df_test_chgoff, df_test_good], axis=1)
df_val_eval = pd.concat([df_val_chgoff, df_val_good], axis=1)

In [ ]:
df_test_eval["norm_const"] = df_test_eval.apply(lambda x : x["chgoff_dens"]*pp_chgoff + x["PIF_dens"]*pp_PIF, axis = 1)

In [ ]:
df_test_eval["prob_chgoff"] = df_test_eval.apply(lambda x : (x["chgoff_dens"]*pp_chgoff)/x["norm_const"] , axis = 1)
df_test_eval["prob_PIF"] = df_test_eval.apply(lambda x : (x["PIF_dens"]*pp_PIF)/x["norm_const"] , axis = 1)

In [ ]:
df_val_eval["norm_const"] = df_val_eval.apply(lambda x : x["chgoff_dens"]*pp_chgoff + x["PIF_dens"]*pp_PIF, axis = 1)
df_val_eval["prob_chgoff"] = df_val_eval.apply(lambda x : (x["chgoff_dens"]*pp_chgoff)/x["norm_const"] , axis = 1)
df_val_eval["prob_PIF"] = df_val_eval.apply(lambda x : (x["PIF_dens"]*pp_PIF)/x["norm_const"] , axis = 1)

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_val.csv"
dfval = pd.read_csv(fp)
df_val_eval["LoanStatus"] = dfval.LoanStatus

In [ ]:
sel_chgoff = df_val_eval.LoanStatus == 1
df_val_eval_chgoff = df_val_eval[sel_chgoff]

In [ ]:
from sklearn.metrics import precision_recall_curve,auc

In [ ]:
y_scores = df_val_eval["prob_chgoff"]

In [ ]:
# Calculate precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(df_val_eval.LoanStatus, y_scores)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(thresholds, precision[:-1], 'b--', label='Precision')
plt.plot(thresholds, recall[:-1], 'r--', label='Recall')
plt.xlabel('Threshold')
plt.legend(loc='lower left')
plt.ylim([0,1])
plt.grid(True)

In [ ]:
from sklearn.calibration import CalibrationDisplay
disp = CalibrationDisplay.from_predictions(df_val_eval.LoanStatus, y_scores)
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
deciles = [ i/10 for i in range(10)]
pchgoff_res = {"decile": deciles, "pchgoff": np.quantile(y_scores, deciles)} 
df_prob_quantiles = pd.DataFrame.from_dict(pchgoff_res, orient="columns")

In [ ]:
TN = len(precision) -5
pTN = precision[:TN]
rTN = recall[:TN]
tTN = thresholds[:TN]
df_th_res = pd.DataFrame.from_dict({"thresh": tTN, "precision": pTN, "recall": rTN}, orient="columns")

In [ ]:
sel_int_region = (df_th_res.recall >=0.8) & (df_th_res.recall < 0.83)  

In [ ]:
df_th_res[sel_int_region]

In [ ]:
THSEL = 0.031

In [ ]:
df_test_eval["prediction"] = df_test_eval["prob_chgoff"].apply(lambda x: 0 if x < THSEL else 1)

In [ ]:
df_test_eval["LoanStatus"] = test_loan_status

In [ ]:
df_test_eval

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(df_test_eval.LoanStatus, df_test_eval.prediction))